# Experiment: ML Generation and Convergence

Objective:
- Generate `ml_a` (SDXL) and `ml_b` (PixArt-alpha) covers from `generation_prompts.csv`.
- Run a small inference-step sweep and inspect a simple convergence proxy.
- Optionally run full generation and merge with real covers.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


def read_rows_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', newline='', encoding='utf-8') as f:
        return [dict(r) for r in csv.DictReader(f)]


def count_rows(path: Path) -> int:
    return len(read_rows_csv(path))


## Parameters

Use `engine="stub"` for fast local checks. Switch to `engine="diffusers"` for real model generation.

In [ ]:
from src.data.generate_ml_covers import generate_ml_covers_from_prompts
from src.data.images import load_image
from src.data.merge_covers_master import merge_covers_master
import numpy as np
import matplotlib.pyplot as plt

PROMPTS_CSV = PROJECT_ROOT / 'data/manifests/generation_prompts.csv'
REAL_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_real.csv'
ML_A_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_a.csv'
ML_B_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_b.csv'
MERGED_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master.csv'

ENGINE = 'stub'  # 'stub' or 'diffusers'
SWEEP_STEPS = [10, 20, 30]
GUIDANCE_SCALE = 7.0
MAX_GROUPS_SWEEP = 12
RUN_FULL_GENERATION = False
MERGE_AFTER_FULL_GENERATION = False

print(f'Prompts CSV exists: {PROMPTS_CSV.exists()}')
print(f'Engine: {ENGINE}')


## Convergence Sweep (Small Subset)

This runs generation multiple times with different `num_inference_steps`, then compares image MSE between consecutive step counts for a fixed sample group.

In [ ]:
sweep_rows: list[dict[str, object]] = []
prev_images: dict[str, np.ndarray] = {}
sample_group_id: int | None = None

for steps in SWEEP_STEPS:
    outputs = generate_ml_covers_from_prompts(
        project_root=PROJECT_ROOT,
        prompts_csv=PROMPTS_CSV,
        engine=ENGINE,
        num_inference_steps=steps,
        guidance_scale=GUIDANCE_SCALE,
        max_groups=MAX_GROUPS_SWEEP,
        seed_base=42,
    )

    rows_a = read_rows_csv(outputs['covers_master_ml_a'])
    rows_b = read_rows_csv(outputs['covers_master_ml_b'])
    if not rows_a or not rows_b:
        raise RuntimeError('Generation produced empty manifests.')

    if sample_group_id is None:
        sample_group_id = int(rows_a[0]['group_id'])

    for source, rows in [('ml_a', rows_a), ('ml_b', rows_b)]:
        row = next(r for r in rows if int(r['group_id']) == sample_group_id)
        img = np.asarray(load_image(PROJECT_ROOT / row['image_path']), dtype=np.float32)

        mse_to_prev = None
        if source in prev_images:
            mse_to_prev = float(np.mean((img - prev_images[source]) ** 2))
        prev_images[source] = img

        sweep_rows.append(
            {
                'steps': steps,
                'source': source,
                'group_id': sample_group_id,
                'pixel_mean': float(img.mean()),
                'pixel_std': float(img.std()),
                'mse_to_previous_steps': mse_to_prev,
            }
        )

print('Sweep results (first rows):')
for row in sweep_rows[:6]:
    print(row)


In [ ]:
plot_rows = [r for r in sweep_rows if r['mse_to_previous_steps'] is not None]
if plot_rows:
    plt.figure(figsize=(8, 4))
    for source in ('ml_a', 'ml_b'):
        xs = [int(r['steps']) for r in plot_rows if r['source'] == source]
        ys = [float(r['mse_to_previous_steps']) for r in plot_rows if r['source'] == source]
        plt.plot(xs, ys, marker='o', label=source)

    plt.title('Convergence proxy: MSE to previous inference-step run')
    plt.xlabel('num_inference_steps')
    plt.ylabel('MSE to previous step')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print('No MSE plot points available yet.')


## Optional Full Generation Run

Set `RUN_FULL_GENERATION=True` to generate all groups (`500`).

In [ ]:
if RUN_FULL_GENERATION:
    outputs = generate_ml_covers_from_prompts(
        project_root=PROJECT_ROOT,
        prompts_csv=PROMPTS_CSV,
        engine=ENGINE,
        num_inference_steps=30,
        guidance_scale=GUIDANCE_SCALE,
        max_groups=None,
        seed_base=42,
    )
    print('Full generation outputs:')
    for k, v in outputs.items():
        print(f'  {k}: {v}')

    if MERGE_AFTER_FULL_GENERATION:
        merged = merge_covers_master(
            project_root=PROJECT_ROOT,
            real_manifest=REAL_MANIFEST,
            ml_a_manifest=ML_A_MANIFEST,
            ml_b_manifest=ML_B_MANIFEST,
            output_manifest=MERGED_MANIFEST,
            expected_groups=500,
        )
        print(f'Merged manifest written: {merged}')
else:
    print('RUN_FULL_GENERATION=False. Skipping full generation.')
